# 07 - Advanced PyTorch Constructs

This notebook mirrors the spirit of the advanced Keras notebook but uses PyTorch patterns.

## Covered ideas
- custom dataset wrapper
- custom transform / augmentation
- custom dropout module
- custom normalization layer
- residual model block
- custom loss
- custom optimizer
- custom scheduler
- explicit training loop
- TensorBoard logging
- post-step weight constraints

The implementation uses FashionMNIST to keep the runtime friendly for Colab.


In [ ]:
# Colab setup
%pip -q install -U torch torchvision tensorboard pandas matplotlib


In [ ]:
import os
import copy
import math
import random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter
from torchvision import datasets, transforms

SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


## Custom dataset and transform


In [ ]:
class AdditiveGaussianNoise:
    def __init__(self, std=0.05):
        self.std = std

    def __call__(self, x):
        return torch.clamp(x + torch.randn_like(x) * self.std, 0.0, 1.0)

class RandomIntensityScale:
    def __init__(self, low=0.8, high=1.2):
        self.low = low
        self.high = high

    def __call__(self, x):
        scale = random.uniform(self.low, self.high)
        return torch.clamp(x * scale, 0.0, 1.0)

base_transform = transforms.ToTensor()
aug_transform = transforms.Compose([
    transforms.ToTensor(),
    AdditiveGaussianNoise(0.05),
    RandomIntensityScale(0.85, 1.15),
])

class WrappedFashionMNIST(Dataset):
    def __init__(self, train=True, transform=None, limit=None):
        self.dataset = datasets.FashionMNIST(root="data", train=train, download=True)
        self.transform = transform
        self.indices = list(range(len(self.dataset)))
        if limit is not None:
            self.indices = self.indices[:limit]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.dataset[self.indices[idx]]
        if self.transform is not None:
            image = self.transform(image)
        return image, label


In [ ]:
train_full = WrappedFashionMNIST(train=True, transform=aug_transform, limit=15000)
test_set = WrappedFashionMNIST(train=False, transform=base_transform, limit=3000)

train_len = 12000
val_len = len(train_full) - train_len
train_set, val_set = random_split(train_full, [train_len, val_len], generator=torch.Generator().manual_seed(SEED))

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False)
test_loader = DataLoader(test_set, batch_size=256, shuffle=False)

len(train_set), len(val_set), len(test_set)


## Custom modules


In [ ]:
class GaussianFeatureDropout(nn.Module):
    def __init__(self, rate=0.2):
        super().__init__()
        self.rate = rate

    def forward(self, x):
        if self.training:
            noise = torch.empty_like(x).uniform_(1.0 - self.rate, 1.0 + self.rate)
            return x * noise
        return x

class FeatureLayerNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.gamma = nn.Parameter(torch.ones(dim))
        self.beta = nn.Parameter(torch.zeros(dim))
        self.eps = eps

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = ((x - mean) ** 2).mean(dim=-1, keepdim=True)
        x_norm = (x - mean) / torch.sqrt(var + self.eps)
        return x_norm * self.gamma + self.beta

class ResidualMLPBlock(nn.Module):
    def __init__(self, dim, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.norm1 = FeatureLayerNorm(dim)
        self.drop = GaussianFeatureDropout(dropout)
        self.fc2 = nn.Linear(dim, dim)
        self.norm2 = FeatureLayerNorm(dim)

    def forward(self, x):
        residual = x
        x = F.leaky_relu(self.norm1(self.fc1(x)), negative_slope=0.1)
        x = self.drop(x)
        x = self.norm2(self.fc2(x))
        return F.relu(x + residual)

class ResidualClassifier(nn.Module):
    def __init__(self, dim=256, num_classes=10):
        super().__init__()
        self.flatten = nn.Flatten()
        self.input_proj = nn.Linear(28 * 28, dim)
        self.block1 = ResidualMLPBlock(dim, dropout=0.2)
        self.block2 = ResidualMLPBlock(dim, dropout=0.2)
        self.out = nn.Linear(dim, num_classes)

    def forward(self, x):
        x = self.flatten(x)
        x = F.leaky_relu(self.input_proj(x), negative_slope=0.1)
        x = self.block1(x)
        x = self.block2(x)
        return self.out(x)


In [ ]:
class LabelSmoothedCrossEntropy(nn.Module):
    def __init__(self, smoothing=0.1):
        super().__init__()
        self.smoothing = smoothing

    def forward(self, logits, target):
        n_classes = logits.size(1)
        log_probs = F.log_softmax(logits, dim=1)
        with torch.no_grad():
            true_dist = torch.zeros_like(log_probs)
            true_dist.fill_(self.smoothing / (n_classes - 1))
            true_dist.scatter_(1, target.unsqueeze(1), 1.0 - self.smoothing)
        return torch.mean(torch.sum(-true_dist * log_probs, dim=1))


## Custom optimizer and scheduler


In [ ]:
class SimpleMomentumTorch(torch.optim.Optimizer):
    def __init__(self, params, lr=1e-3, momentum=0.9):
        defaults = dict(lr=lr, momentum=momentum)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self, closure=None):
        loss = None
        if closure is not None:
            with torch.enable_grad():
                loss = closure()

        for group in self.param_groups:
            lr = group["lr"]
            momentum = group["momentum"]

            for p in group["params"]:
                if p.grad is None:
                    continue
                grad = p.grad
                state = self.state[p]
                if "velocity" not in state:
                    state["velocity"] = torch.zeros_like(p)
                velocity = state["velocity"]
                velocity.mul_(momentum).add_(grad, alpha=-lr)
                p.add_(velocity)
        return loss

def constrain_positive_output_weights(model):
    with torch.no_grad():
        model.out.weight.data.clamp_(min=0.0)


In [ ]:
model = ResidualClassifier(dim=256).to(device)
criterion = LabelSmoothedCrossEntropy(smoothing=0.05)
optimizer = SimpleMomentumTorch(model.parameters(), lr=8e-4, momentum=0.9)
scheduler = torch.optim.lr_scheduler.LambdaLR(
    optimizer,
    lr_lambda=lambda epoch: 1.0 if epoch < 3 else max(0.2, 0.95 ** (epoch - 3))
)

writer = SummaryWriter(log_dir=os.path.join("runs", "part2_pytorch_advanced"))


In [ ]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()
    total, correct = 0, 0
    losses = []
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        logits = model(x)
        loss = criterion(logits, y)
        losses.append(loss.item())
        preds = logits.argmax(dim=1)
        correct += (preds == y).sum().item()
        total += y.size(0)
    return np.mean(losses), correct / total


## Training loop


In [ ]:
best_state = None
best_val_acc = -1
history = {"train_loss": [], "val_loss": [], "val_acc": [], "lr": []}
patience = 4
patience_counter = 0

for epoch in range(12):
    model.train()
    batch_losses = []

    for x, y in train_loader:
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()
        logits = model(x)
        loss = criterion(logits, y)
        loss.backward()
        optimizer.step()
        constrain_positive_output_weights(model)
        batch_losses.append(loss.item())

    scheduler.step()

    train_loss = float(np.mean(batch_losses))
    val_loss, val_acc = evaluate(model, val_loader)
    current_lr = scheduler.get_last_lr()[0]

    history["train_loss"].append(train_loss)
    history["val_loss"].append(val_loss)
    history["val_acc"].append(val_acc)
    history["lr"].append(current_lr)

    writer.add_scalar("loss/train", train_loss, epoch)
    writer.add_scalar("loss/val", val_loss, epoch)
    writer.add_scalar("acc/val", val_acc, epoch)
    writer.add_scalar("lr", current_lr, epoch)

    print(
        f"epoch={epoch+1} train_loss={train_loss:.4f} "
        f"val_loss={val_loss:.4f} val_acc={val_acc:.4f} lr={current_lr:.6f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_state = copy.deepcopy(model.state_dict())
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= patience:
            print("Early stopping triggered.")
            break

model.load_state_dict(best_state)
writer.close()


In [ ]:
test_loss, test_acc = evaluate(model, test_loader)
pd.DataFrame([{
    "best_val_acc": best_val_acc,
    "test_loss": test_loss,
    "test_acc": test_acc,
}])


In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(history["val_acc"], label="val_acc")
plt.plot(history["lr"], label="learning_rate")
plt.title("Advanced PyTorch run")
plt.xlabel("Epoch")
plt.legend()
plt.show()


In [ ]:
print("TensorBoard log root:", os.path.abspath("runs/part2_pytorch_advanced"))
# In Colab:
# %load_ext tensorboard
# %tensorboard --logdir runs/part2_pytorch_advanced


## Wrap-up

This notebook demonstrates advanced PyTorch ideas in one clean training pipeline:

- custom dataset wrapper and transforms
- custom loss
- custom dropout and normalization
- residual architecture
- custom optimizer
- custom scheduler
- explicit training loop
- TensorBoard logging
- post-step constraints

For the video, emphasize *why* each custom object exists instead of treating the notebook like a sacred scroll that must be recited line by line without oxygen.
